# Model Experimentation Notebook

In this notebook, we experiment with multiple baseline models (without hyperparameter tuning)
to identify which algorithm performs best on our fraud detection problem.

**Important:** Since our data is highly imbalanced (0.13% fraud), we will:
- Use `class_weight='balanced'` in our models instead of SMOTE
- Evaluate using Precision, Recall, F1-score, and ROC-AUC — NOT accuracy (which would be misleading)

## Step 1: Load Train/Test Data

We load the pre-split, feature-engineered train and test sets created earlier by our pipeline scripts (`build_features.py` and `split_data.py`).

In [2]:
import pandas as pd

# Load the train/test splits saved earlier
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Fraud ratio in y_train:", y_train.mean())
print("Fraud ratio in y_test:", y_test.mean())

X_train shape: (5090096, 11)
X_test shape: (1272524, 11)
Fraud ratio in y_train: 0.0012907418642005967
Fraud ratio in y_test: 0.0012911347840983745


## Step 2 Baseline Model — Logistic Regression (with MLflow Tracking)

We re-run our Logistic Regression baseline, this time logging parameters,
metrics, and the trained model to MLflow — so we can compare all experiments
in one place later using `mlflow ui`.

In [9]:
import mlflow

# Use SQLite backend instead of file-based storage (avoids maintenance mode error)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [10]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score

# Set experiment name (creates it if it doesn't exist)
mlflow.set_experiment("fraud-detection-baseline-models")

with mlflow.start_run(run_name="logistic_regression_baseline"):
    # Train model
    log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    log_reg.fit(X_train, y_train)

    # Predictions
    y_pred = log_reg.predict(X_test)
    y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)

    # Log metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # Log the trained model itself
    mlflow.sklearn.log_model(log_reg, "model")

    # Print results as before
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("ROC-AUC Score:", roc_auc)

2026/07/30 17:35:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/30 17:35:33 INFO mlflow.store.db.utils: Updating database tables
2026/07/30 17:35:38 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-baseline-models' does not exist. Creating a new experiment.
2026/07/30 17:36:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97   1270881
           1       0.02      0.98      0.04      1643

    accuracy                           0.94   1272524
   macro avg       0.51      0.96      0.50   1272524
weighted avg       1.00      0.94      0.97   1272524

Confusion Matrix:
[[1190032   80849]
 [     35    1608]]
ROC-AUC Score: 0.99068662445439


## Step 3: Baseline Model — Random Forest (with MLflow Tracking)

We now train Random Forest as our second baseline model, logging it to the
same MLflow experiment so we can compare it against Logistic Regression.

In [11]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="random_forest_baseline"):

    # Train model
    rf_model = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        max_depth=10
    )
    rf_model.fit(X_train, y_train)

    # Predictions
    y_pred_rf = rf_model.predict(X_test)
    y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision_rf = precision_score(y_test, y_pred_rf)
    recall_rf = recall_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)
    roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)

    # Log metrics
    mlflow.log_metric("precision", precision_rf)
    mlflow.log_metric("recall", recall_rf)
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)

    # Log the trained model
    mlflow.sklearn.log_model(rf_model, "model")

    # Print results
    print("Classification Report (Random Forest):")
    print(classification_report(y_test, y_pred_rf))
    print("Confusion Matrix (Random Forest):")
    print(confusion_matrix(y_test, y_pred_rf))
    print("ROC-AUC Score (Random Forest):", roc_auc_rf)

2026/07/30 17:47:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       1.00      0.99      0.99   1270881
           1       0.08      0.99      0.15      1643

    accuracy                           0.99   1272524
   macro avg       0.54      0.99      0.57   1272524
weighted avg       1.00      0.99      0.99   1272524

Confusion Matrix (Random Forest):
[[1251868   19013]
 [     10    1633]]
ROC-AUC Score (Random Forest): 0.9992764279660398


## Step 4: Baseline Model — XGBoost (with MLflow Tracking)

We now try XGBoost, a gradient boosting algorithm that is widely considered
one of the best performers for imbalanced classification problems like fraud detection.

Instead of `class_weight='balanced'`, XGBoost uses `scale_pos_weight` — a ratio
that tells the model how much more to focus on the minority (fraud) class.

In [13]:
import mlflow.xgboost

In [14]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight: ratio of negative class to positive class
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight value:", scale_pos_weight)

with mlflow.start_run(run_name="xgboost_baseline"):

    # Train model
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)

    # Predictions
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision_xgb = precision_score(y_test, y_pred_xgb)
    recall_xgb = recall_score(y_test, y_pred_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

    # Log parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)

    # Log metrics
    mlflow.log_metric("precision", precision_xgb)
    mlflow.log_metric("recall", recall_xgb)
    mlflow.log_metric("f1_score", f1_xgb)
    mlflow.log_metric("roc_auc", roc_auc_xgb)

    # Log the trained model
    mlflow.xgboost.log_model(xgb_model, "model")

    # Print results
    print("Classification Report (XGBoost):")
    print(classification_report(y_test, y_pred_xgb))
    print("Confusion Matrix (XGBoost):")
    print(confusion_matrix(y_test, y_pred_xgb))
    print("ROC-AUC Score (XGBoost):", roc_auc_xgb)

scale_pos_weight value: 773.7482496194825


2026/07/30 17:58:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classification Report (XGBoost):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.52      0.99      0.68      1643

    accuracy                           1.00   1272524
   macro avg       0.76      0.99      0.84   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion Matrix (XGBoost):
[[1269351    1530]
 [     17    1626]]
ROC-AUC Score (XGBoost): 0.9994933173494438


## Step 5  Hyperparameter Tuning — XGBoost with Optuna (Persistent Storage + Pruning)

This time, we save the Optuna study to a SQLite database file so it persists
even if the notebook variable is accidentally overwritten or the kernel restarts.
We also add proper pruning using XGBoost's native API, so poor-performing trials
stop early instead of running to completion — saving significant time.

In [10]:
import optuna
import xgboost as xgb
from optuna.integration import XGBoostPruningCallback
from sklearn.model_selection import train_test_split

# Create validation split from training data for tuning
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

dtrain = xgb.DMatrix(X_train_sub, label=y_train_sub)
dval = xgb.DMatrix(X_val, label=y_val)


def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 50, 800),
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'seed': 42
    }

    n_estimators = trial.suggest_int('n_estimators', 100, 400)

    pruning_callback = XGBoostPruningCallback(trial, 'validation-logloss')

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=n_estimators,
        evals=[(dval, 'validation')],
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    y_pred_proba = booster.predict(dval)
    y_pred = (y_pred_proba >= 0.5).astype(int)
    f1 = f1_score(y_val, y_pred)

    return f1


# Save study to a SQLite file so it PERSISTS even if variable gets overwritten
storage_name = "sqlite:///optuna_study.db"

study = optuna.create_study(
    study_name="xgboost_fraud_tuning",
    direction='maximize',
    storage=storage_name,
    load_if_exists=True  # if it already exists, resume instead of overwriting
)

study.optimize(objective, n_trials=30)

print("Best F1-score:", study.best_value)
print("Best parameters:", study.best_params)

c:\Users\Ali Raza\Desktop\fraud-transaction-detection\venv\Lib\site-packages\optuna\integration\xgboost.py:14: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  optuna_warn(f"{msg} Use `optuna_integration.xgboost` instead.", FutureWarning)
[I 2026-07-31 10:05:24,483] A new study created in RDB with name: xgboost_fraud_tuning
[I 2026-07-31 10:06:44,456] Trial 0 finished with value: 0.19029906263948818 and parameters: {'max_depth': 5, 'learning_rate': 0.018858958272596763, 'subsample': 0.9806880098285207, 'colsample_bytree': 0.9682043078655018, 'scale_pos_weight': 211.95945287489258, 'n_estimators': 109}. Best is trial 0 with value: 0.19029906263948818.
[I 2026-07-31 10:11:04,877] Trial 1 finished with value: 0.7889727600918937 and parameters: {'max_depth': 10, 'learning_rate': 0.15130877533531048, 'subsample': 0.7552821325

KeyboardInterrupt: 

## Step 6: Verify Best Parameters — Train on Full Data, Test on X_test

Before moving to production code, we do a quick sanity check in the notebook:
train a fresh model on the FULL training data using Optuna's best parameters,
and evaluate it on the untouched test set.

In [11]:
# Get the best parameters found by Optuna
best_params = study.best_params
print("Best parameters from Optuna:")
print(best_params)
print("Best F1-score (on validation subset):", study.best_value)

Best parameters from Optuna:
{'max_depth': 10, 'learning_rate': 0.15130877533531048, 'subsample': 0.755282132544784, 'colsample_bytree': 0.7713453064298333, 'scale_pos_weight': 314.9867636898015, 'n_estimators': 287}
Best F1-score (on validation subset): 0.7889727600918937


## Step 7: Train Final Model on Full Data, Evaluate on Test Set

Using the best parameters found by Optuna, we train a fresh model on the
FULL training data (not the subset used during tuning), then evaluate on
the untouched test set to get a reliable performance estimate.

In [ ]:
# Train final model on FULL training data using best parameters

final_model = XGBClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
final_model.fit(X_train, y_train)

# Predictions on test set
y_pred_final = final_model.predict(X_test)
y_pred_proba_final = final_model.predict_proba(X_test)[:, 1]

NameError: name 'precision_score' is not defined

In [13]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
# Calculate metrics
precision_final = precision_score(y_test, y_pred_final)
recall_final = recall_score(y_test, y_pred_final)
f1_final = f1_score(y_test, y_pred_final)
roc_auc_final = roc_auc_score(y_test, y_pred_proba_final)

# Print results
print("Classification Report (Tuned XGBoost, Full Train, Test Set):")
print(classification_report(y_test, y_pred_final))
print("Confusion Matrix (Tuned XGBoost):")
print(confusion_matrix(y_test, y_pred_final))
print("ROC-AUC Score (Tuned XGBoost):", roc_auc_final)

Classification Report (Tuned XGBoost, Full Train, Test Set):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.69      0.93      0.79      1643

    accuracy                           1.00   1272524
   macro avg       0.85      0.97      0.90   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion Matrix (Tuned XGBoost):
[[1270196     685]
 [    108    1535]]
ROC-AUC Score (Tuned XGBoost): 0.9988762809361795


## Step 8: Threshold Tuning

Instead of using the default 0.5 cutoff, we test different probability thresholds
to find a better balance between Precision and Recall — specifically trying to
reduce False Negatives (missed fraud cases) without letting False Positives
grow too much.

In [14]:
import numpy as np

# Try different thresholds and see how precision/recall change
thresholds = np.arange(0.1, 0.6, 0.05)

print(f"{'Threshold':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'FN':<8}{'FP':<8}")
print("-" * 60)

for t in thresholds:
    y_pred_t = (y_pred_proba_final >= t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    cm = confusion_matrix(y_test, y_pred_t)
    fn = cm[1][0]
    fp = cm[0][1]
    print(f"{t:<12.2f}{p:<12.4f}{r:<12.4f}{f1:<12.4f}{fn:<8}{fp:<8}")

Threshold   Precision   Recall      F1          FN      FP      
------------------------------------------------------------
0.10        0.5905      0.9714      0.7345      47      1107    
0.15        0.6164      0.9653      0.7524      57      987     
0.20        0.6336      0.9598      0.7633      66      912     
0.25        0.6479      0.9531      0.7714      77      851     
0.30        0.6623      0.9501      0.7805      82      796     
0.35        0.6707      0.9458      0.7848      89      763     
0.40        0.6778      0.9422      0.7884      95      736     
0.45        0.6852      0.9367      0.7915      104     707     
0.50        0.6914      0.9343      0.7947      108     685     
0.55        0.6987      0.9300      0.7979      115     659     


In [15]:
# Test even lower thresholds to see if we can get FN closer to baseline (17)
thresholds_lower = np.arange(0.01, 0.11, 0.01)

print(f"{'Threshold':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'FN':<8}{'FP':<8}")
print("-" * 60)

for t in thresholds_lower:
    y_pred_t = (y_pred_proba_final >= t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    cm = confusion_matrix(y_test, y_pred_t)
    fn = cm[1][0]
    fp = cm[0][1]
    print(f"{t:<12.2f}{p:<12.4f}{r:<12.4f}{f1:<12.4f}{fn:<8}{fp:<8}")

Threshold   Precision   Recall      F1          FN      FP      
------------------------------------------------------------
0.01        0.4276      0.9890      0.5971      18      2175    
0.02        0.4755      0.9860      0.6416      23      1787    
0.03        0.5045      0.9836      0.6669      27      1587    
0.04        0.5263      0.9799      0.6848      33      1449    
0.05        0.5427      0.9787      0.6982      35      1355    
0.06        0.5562      0.9781      0.7092      36      1282    
0.07        0.5668      0.9757      0.7171      40      1225    
0.08        0.5736      0.9720      0.7215      46      1187    
0.09        0.5852      0.9720      0.7306      46      1132    
0.10        0.5905      0.9714      0.7345      47      1107    


## Step 9: Hyperparameter Tuning — Optuna Optimizing for F2-score

We rerun Optuna, but this time optimizing for F2-score instead of F1-score.
F2-score weights Recall twice as much as Precision, which better aligns with
our business goal: minimizing missed fraud cases (False Negatives), even if
it means accepting somewhat more false alarms.

We use a NEW study name and NEW database file to keep this separate from
our previous F1-optimized study.

In [16]:
import optuna
import xgboost as xgb
from optuna.integration import XGBoostPruningCallback
from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split

# Reuse the same validation split from before
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

dtrain = xgb.DMatrix(X_train_sub, label=y_train_sub)
dval = xgb.DMatrix(X_val, label=y_val)


def objective_f2(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 50, 800),
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'seed': 42
    }

    n_estimators = trial.suggest_int('n_estimators', 100, 400)

    pruning_callback = XGBoostPruningCallback(trial, 'validation-logloss')

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=n_estimators,
        evals=[(dval, 'validation')],
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    y_pred_proba = booster.predict(dval)
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # F2-score: weights Recall 2x more than Precision (beta=2)
    f2 = fbeta_score(y_val, y_pred, beta=2)

    return f2


# New study, saved to a NEW database file (keeps this separate from F1 study)
storage_name_f2 = "sqlite:///optuna_study_f2.db"

study_f2 = optuna.create_study(
    study_name="xgboost_fraud_tuning_f2",
    direction='maximize',
    storage=storage_name_f2,
    load_if_exists=True
)

study_f2.optimize(objective_f2, n_trials=30)

print("Best F2-score:", study_f2.best_value)
print("Best parameters:", study_f2.best_params)

[I 2026-07-31 10:48:06,259] A new study created in RDB with name: xgboost_fraud_tuning_f2
[I 2026-07-31 10:49:53,788] Trial 0 finished with value: 0.36413013356980545 and parameters: {'max_depth': 5, 'learning_rate': 0.0503405418890761, 'subsample': 0.733118826615768, 'colsample_bytree': 0.8930958575617052, 'scale_pos_weight': 797.3341463366639, 'n_estimators': 151}. Best is trial 0 with value: 0.36413013356980545.
[I 2026-07-31 10:54:01,704] Trial 1 finished with value: 0.8466161484191272 and parameters: {'max_depth': 9, 'learning_rate': 0.04024002876092483, 'subsample': 0.9287785379436121, 'colsample_bytree': 0.6334229051857528, 'scale_pos_weight': 62.578776638611785, 'n_estimators': 311}. Best is trial 1 with value: 0.8466161484191272.
[I 2026-07-31 10:55:52,395] Trial 2 finished with value: 0.8323743766781742 and parameters: {'max_depth': 7, 'learning_rate': 0.19523745690985952, 'subsample': 0.6078254793140245, 'colsample_bytree': 0.7757493486926099, 'scale_pos_weight': 368.6740687

AssertionError: Should not reach.

In [17]:
print("Best F2-score:", study_f2.best_value)
print("Best parameters:", study_f2.best_params)

Best F2-score: 0.8466161484191272
Best parameters: {'max_depth': 9, 'learning_rate': 0.04024002876092483, 'subsample': 0.9287785379436121, 'colsample_bytree': 0.6334229051857528, 'scale_pos_weight': 62.578776638611785, 'n_estimators': 311}


## Step 10: Train F2-Optimized Model on Full Data, Evaluate on Test Set

Using the best parameters found by F2-score optimization, we train a fresh
model on the full training data and evaluate on the test set — comparing
against both the baseline XGBoost and the F1-tuned XGBoost.

In [18]:
# Get best parameters from F2 study
best_params_f2 = study_f2.best_params
print("Best F2 parameters:")
print(best_params_f2)

# Train final model on FULL training data using F2-optimized parameters
final_model_f2 = XGBClassifier(
    **best_params_f2,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
final_model_f2.fit(X_train, y_train)

# Predictions on test set
y_pred_f2 = final_model_f2.predict(X_test)
y_pred_proba_f2 = final_model_f2.predict_proba(X_test)[:, 1]

# Calculate metrics
precision_f2 = precision_score(y_test, y_pred_f2)
recall_f2 = recall_score(y_test, y_pred_f2)
f1_f2 = f1_score(y_test, y_pred_f2)
roc_auc_f2 = roc_auc_score(y_test, y_pred_proba_f2)

# Print results
print("Classification Report (F2-Tuned XGBoost, Full Train, Test Set):")
print(classification_report(y_test, y_pred_f2))
print("Confusion Matrix (F2-Tuned XGBoost):")
print(confusion_matrix(y_test, y_pred_f2))
print("ROC-AUC Score (F2-Tuned XGBoost):", roc_auc_f2)

Best F2 parameters:
{'max_depth': 9, 'learning_rate': 0.04024002876092483, 'subsample': 0.9287785379436121, 'colsample_bytree': 0.6334229051857528, 'scale_pos_weight': 62.578776638611785, 'n_estimators': 311}
Classification Report (F2-Tuned XGBoost, Full Train, Test Set):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.56      0.98      0.71      1643

    accuracy                           1.00   1272524
   macro avg       0.78      0.99      0.85   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion Matrix (F2-Tuned XGBoost):
[[1269587    1294]
 [     25    1618]]
ROC-AUC Score (F2-Tuned XGBoost): 0.9995974871348885
